# Lab 4 — Setting Up and Running Many-Objective Evolutionary Optimization

## Background

In the previous labs, you *sampled* the lever space to find candidate policies. In this lab
you **search** it using a **Many-Objective Evolutionary Algorithm (MOEA)**, which iteratively
evolves a population of solutions toward the Pareto-optimal front.

### Pareto optimality

A solution $\mathbf{x}$ is **Pareto-dominated** by $\mathbf{x}'$ if $\mathbf{x}'$ is at
least as good on all objectives and strictly better on at least one. The **Pareto-approximate
set** (or Pareto front) is the set of all non-dominated solutions found during the search.

### ε-NSGA-II: the default algorithm

The workbench uses **ε-NSGA-II** (Non-dominated Sorting Genetic Algorithm II with ε-dominance).
Key idea: instead of standard Pareto dominance, a grid of cell size **ε** is imposed on
the objective space. Per cell, only one solution is retained — the one closest to the cell
origin. This prevents the archive from growing unboundedly and promotes diversity.

Choosing **ε** involves a trade-off:
- **Large ε** → coarser grid → fewer solutions, faster convergence, lower precision.
- **Small ε** → finer grid → more solutions, slower convergence, higher precision.

### How many function evaluations (NFE)?

The algorithm terminates after a user-specified number of function evaluations. Choosing
the right NFE requires convergence analysis (Lab 5). As a rule of thumb:
- Start with 5 000 NFE for exploration.
- Use convergence metrics (hypervolume, ε-progress) to judge if more NFE is needed.
- In practice, 50 000–250 000 NFE is common for problems with 4 objectives and 100 levers.

### What you will do

1. Reconfigure the lake model for optimisation (specify `kind` for each outcome).
2. Run a first optimisation with 5 000 NFE and visualise the Pareto front.
3. Explore the effect of different ε values on the number and distribution of solutions.
4. Add convergence tracking via `ArchiveLogger` and `EpsilonProgress`.
5. Run two optimisations (5 000 and 100 000 NFE) and compare convergence signals.

## 1. Model setup for optimisation

Two changes from the exploratory model:
1. Each outcome needs a `kind`: `MINIMIZE` or `MAXIMIZE`.
2. We add `expected_range` to each outcome — required for hypervolume calculation (Lab 5).
3. `Constant` objects fix parameters that should not vary during optimisation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from lakemodel_function import lake_problem
from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           MultiprocessingEvaluator, Constant, ema_logging)
from ema_workbench.analysis import parcoords

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Lake model for optimisation ---
lake_model = Model('lakeproblem', function=lake_problem)
lake_model.time_horizon = 100

lake_model.uncertainties = [
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('delta', 0.93,  0.99),
]
lake_model.levers = [RealParameter(f'l{i}', 0, 0.1)
                     for i in range(lake_model.time_horizon)]

# Outcomes with optimisation direction AND expected range (needed for hypervolume)
lake_model.outcomes = [
    ScalarOutcome('max_P',       kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome('utility',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('inertia',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('reliability', kind=ScalarOutcome.MAXIMIZE),
]

# Fix nsamples and alpha as constants so the optimiser doesn't vary them
lake_model.constants = [
    Constant('alpha',    0.41),
    Constant('nsamples', 150),
]

## 2. First optimisation run — 3 objectives, 5 000 NFE

We start with a simplified 3-objective formulation (ignoring inertia) to visualise the
Pareto front in 3D. This is purely for illustration — in practice we use all four objectives.

In [ ]:
import tempfile
_tmpdir = tempfile.mkdtemp()

# Temporarily use 3 outcomes (drop inertia) for 3-D visualisation
# Note: in ema_workbench 3.x, assigning outcomes merges rather than replaces,
# so we must explicitly remove inertia from the existing map.
del lake_model.outcomes["inertia"]

with MultiprocessingEvaluator(lake_model) as evaluator:
    results_3obj, _ = evaluator.optimize(nfe=5000, epsilons=[0.25, 0.1, 0.1],
                                                    directory=_tmpdir, filename='tmp_3obj.tar.gz')

print(f'Solutions found: {len(results_3obj)}')

### 2.1 3-D scatter plot of the Pareto front

With three objectives, we can plot the Pareto front directly in 3-D space. Each point is
a non-dominated solution found by the algorithm. Note the clear trade-off surface.

In [ ]:
obj_3 = results_3obj.loc[:, ['max_P', 'utility', 'reliability']]

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(obj_3['max_P'], obj_3['utility'], obj_3['reliability'],
           alpha=0.7, edgecolors='none')
ax.set_xlabel('max P (min)')
ax.set_ylabel('utility (max)')
ax.set_zlabel('reliability (max)')
ax.set_title('Pareto front — 3 objectives, 5 000 NFE')
plt.show()

### 2.2 Parallel coordinates plot

For more than three objectives, **parallel coordinates** (parcoords) are the standard
visualisation. Each vertical axis represents one objective. A line connecting the axes
represents a single Pareto-optimal solution. Crossing lines indicate trade-offs.

We **invert** `max_P` so that "up is better" for all axes (consistent direction of desirability).

In [ ]:
limits = parcoords.get_limits(obj_3)
axes_pc = parcoords.ParallelAxes(limits)
axes_pc.plot(obj_3)
axes_pc.invert_axis('max_P')   # flip so up = better for all axes
plt.title('Pareto front — parallel coordinates (3 objectives)')
plt.show()

## 3. Four-objective formulation

Now restore the full 4-objective model (adding `inertia`). All subsequent analyses use
this formulation.

In [ ]:
# Restore full 4-objective formulation with expected ranges
lake_model.outcomes = [
    ScalarOutcome('max_P',       kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome('utility',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('inertia',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('reliability', kind=ScalarOutcome.MAXIMIZE),
]

## 4. Effect of ε values

The ε parameter controls the granularity of the Pareto-approximate set.
- `ε = [0.5, 0.5, 0.5, 0.5]` → coarse grid → few solutions.
- `ε = [0.125, 0.05, 0.05, 0.05]` → fine grid → many solutions.

We compare both settings at 5 000 NFE.

In [ ]:
import tempfile
_tmpdir = tempfile.mkdtemp()

# --- Coarse epsilons ---
with MultiprocessingEvaluator(lake_model) as evaluator:
    results_coarse, _ = evaluator.optimize(nfe=5000, epsilons=[0.5, 0.5, 0.5, 0.5],
                                                       directory=_tmpdir, filename='tmp_coarse.tar.gz')

obj_coarse = results_coarse.loc[:, ['max_P', 'utility', 'reliability', 'inertia']]
limits_c = parcoords.get_limits(obj_coarse)
axes_c = parcoords.ParallelAxes(limits_c)
axes_c.plot(obj_coarse)
axes_c.invert_axis('max_P')
plt.title(f'Coarse ε = [0.5, 0.5, 0.5, 0.5] — {len(results_coarse)} solutions')
plt.show()

In [ ]:
import tempfile
_tmpdir = tempfile.mkdtemp()

# --- Fine epsilons ---
with MultiprocessingEvaluator(lake_model) as evaluator:
    results_fine, _ = evaluator.optimize(nfe=5000, epsilons=[0.125, 0.05, 0.05, 0.05],
                                                     directory=_tmpdir, filename='tmp_fine.tar.gz')

obj_fine = results_fine.loc[:, ['max_P', 'utility', 'reliability', 'inertia']]
limits_f = parcoords.get_limits(obj_fine)
axes_f = parcoords.ParallelAxes(limits_f)
axes_f.plot(obj_fine)
axes_f.invert_axis('max_P')
plt.title(f'Fine ε = [0.125, 0.05, 0.05, 0.05] — {len(results_fine)} solutions')
plt.show()

print(f'Coarse ε: {len(results_coarse)} solutions')
print(f'Fine ε:   {len(results_fine)} solutions')

## 5. Convergence tracking with ArchiveLogger and EpsilonProgress

To know whether the algorithm has converged, we attach two convergence trackers:

- **`ArchiveLogger`**: saves the full Pareto-approximate archive to disk at every generation.
  These archives can later be used to compute hypervolume and other metrics (Lab 5).
- **`EpsilonProgress`**: counts how many times a new Pareto-grid cell is discovered.
  Early in the search, this count grows quickly; it should plateau when convergence is reached.

We also add a **constraint** on `max_P` to bound the objective space for hypervolume
calculation: no solution with `max_P > 5` is acceptable.

In [ ]:
import os
from ema_workbench import Constraint

os.makedirs('./archives', exist_ok=True)
if os.path.exists('./archives/lab4_run1.tar.gz'):
    os.remove('./archives/lab4_run1.tar.gz')

# Constraint: reject solutions where max_P > 5
constraint_maxP = Constraint('max_pollution', outcome_names='max_P',
                              function=lambda x: max(0, x - 5))

# --- Run with 5 000 NFE (archives saved automatically via filename/directory) ---
with MultiprocessingEvaluator(lake_model) as evaluator:
    results_5k, convergence_5k = evaluator.optimize(
        nfe=5000,
        searchover='levers',
        epsilons=[0.125, 0.05, 0.01, 0.01],
        filename='lab4_run1.tar.gz',
        directory='./archives',
        constraints=[constraint_maxP],
    )

# Save convergence so Lab 5 can load it without re-running
convergence_5k.to_csv('./archives/lab4_convergence_5k.csv', index=False)

print(f'Solutions found (5 000 NFE): {len(results_5k)}')


### 5.1 Quick convergence check with ε-progress

ε-progress counts the number of times the algorithm discovers a new grid cell. A rising,
non-plateauing curve means the search is still finding new regions of the objective space —
i.e., it has NOT converged. A flat plateau indicates convergence.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(convergence_5k['nfe'], convergence_5k['epsilon_progress'])
ax.set_xlabel('Number of function evaluations')
ax.set_ylabel('ε-progress (cumulative)')
ax.set_title('Convergence indicator — 5 000 NFE')
sns.despine()
plt.tight_layout()
plt.show()

### 5.2 Scaling up to 100 000 NFE

Based on the ε-progress curve, 5 000 NFE is typically insufficient. We now run to 100 000 NFE
and save a second archive for comparison in Lab 5.

> **Note:** This run may take 10–30 minutes depending on your machine. The archive is saved to
> `./archives/lab4_run2.tar.gz` and can be loaded in Lab 5 without re-running.

In [ ]:
import os
os.makedirs('./archives', exist_ok=True)
if os.path.exists('./archives/lab4_run2.tar.gz'):
    os.remove('./archives/lab4_run2.tar.gz')

with MultiprocessingEvaluator(lake_model) as evaluator:
    results_100k, convergence_100k = evaluator.optimize(
        nfe=100000,
        searchover='levers',
        epsilons=[0.125, 0.05, 0.01, 0.01],
        filename='lab4_run2.tar.gz',
        directory='./archives',
        constraints=[constraint_maxP],
    )

# Save convergence so Lab 5 can load it without re-running
convergence_100k.to_csv('./archives/lab4_convergence_100k.csv', index=False)

print(f'Solutions found (100 000 NFE): {len(results_100k)}')


In [ ]:
# Compare ε-progress curves between 5k and 100k runs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(convergence_5k['nfe'],   convergence_5k['epsilon_progress'])
axes[0].set_title('5 000 NFE')
axes[0].set_xlabel('NFE'); axes[0].set_ylabel('ε-progress')

axes[1].plot(convergence_100k['nfe'], convergence_100k['epsilon_progress'])
axes[1].set_title('100 000 NFE')
axes[1].set_xlabel('NFE'); axes[1].set_ylabel('ε-progress')

for ax in axes:
    sns.despine(ax=ax)
plt.suptitle('ε-progress: 5k vs 100k function evaluations')
plt.tight_layout()
plt.show()

## Summary

You have:
- Configured a model for multi-objective optimisation with `kind` and `expected_range`.
- Run ε-NSGA-II and visualised the Pareto front in 3-D and via parallel coordinates.
- Explored how ε controls the resolution of the Pareto-approximate set.
- Set up convergence tracking with `ArchiveLogger` and `EpsilonProgress`.
- Observed that 5 000 NFE is insufficient; 100 000 NFE shows signs of convergence.

In **Lab 5** you will load these archives and compute the full suite of convergence metrics,
plus analyse the effect of algorithmic stochasticity across multiple random seeds.